In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import LSTM, Dense
from keras.callbacks import EarlyStopping


In [2]:
data = pd.read_csv("data.csv", header=None)
data.columns = ["EAR", "MAR", "Label"]



print(data["Label"].value_counts())


Label
1    1161
0     443
Name: count, dtype: int64


In [3]:
print(data.value_counts())

EAR       MAR       Label
0.254205  0.621025  1        2
0.194885  0.652353  0        2
0.165165  0.638721  1        2
0.116787  0.641933  1        1
0.218943  0.638517  1        1
                            ..
0.187128  0.662278  1        1
0.187109  0.633547  1        1
0.186975  0.685414  1        1
0.186861  0.621307  1        1
0.307214  0.658154  0        1
Name: count, Length: 1601, dtype: int64


In [4]:
X = data[["EAR", "MAR"]].values
y = data["Label"].values


In [5]:
print(data.value_counts())

EAR       MAR       Label
0.254205  0.621025  1        2
0.194885  0.652353  0        2
0.165165  0.638721  1        2
0.116787  0.641933  1        1
0.218943  0.638517  1        1
                            ..
0.187128  0.662278  1        1
0.187109  0.633547  1        1
0.186975  0.685414  1        1
0.186861  0.621307  1        1
0.307214  0.658154  0        1
Name: count, Length: 1601, dtype: int64


In [ ]:
SEQ_LEN = 30  #frames 

def create_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X, y, SEQ_LEN)

print("X shape:", X_seq.shape)
print("y shape:", y_seq.shape)



X shape: (1574, 30, 2)
y shape: (1574,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42, shuffle=True)


In [8]:
class_weight = {0: 1161 / 443, 1: 1.0}        


In [ ]:
# LSTM model
model = Sequential([LSTM(64, input_shape=(SEQ_LEN, 2)),Dense(32, activation="relu"),Dense(1, activation="sigmoid")])

model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

c:\Users\jyoth\OneDrive\Desktop\DL\mediapipe\mp_env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,265 (75.25 KB)

 Trainable params: 19,265 (75.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:


early_stop = EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True)

history = model.fit(X_train, y_train,validation_data=(X_test, y_test), epochs=20, batch_size=32,class_weight=class_weight,callbacks=[early_stop])


Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.7403 - loss: 0.9846 - val_accuracy: 0.7270 - val_loss: 0.6821
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7403 - loss: 0.9832 - val_accuracy: 0.7270 - val_loss: 0.6789
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6315 - loss: 0.9749 - val_accuracy: 0.7270 - val_loss: 0.6260
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6322 - loss: 0.8876 - val_accuracy: 0.7175 - val_loss: 0.5324
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6195 - loss: 0.8329 - val_accuracy: 0.6571 - val_loss: 0.5640
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6529 - loss: 0.8186 - val_accuracy: 0.7238 - val_loss: 0.5114
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6918 - loss: 0.7969 - val_accuracy: 0.7206 - val_loss: 0.5118
Epoch 8/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6680 - loss: 0.7983 - val_accuracy: 0.7238 - v

In [12]:
model.save("drowsiness_lstm_model.h5")
print("Model saved as drowsiness_lstm_model.h5")


Model saved as drowsiness_lstm_model.h5
